<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 警告：此文件不是训练营教学步骤的一部分
# 这是一个中高级示例，展示了您将学习的内容
# 如果您正在通过训练营学习Chisel，请不要从这里开始
# 请从 [Scala简介](1_intro_to_scala.ipynb) 开始

# Chisel 演示
**下一步：[Scala简介](1_intro_to_scala.ipynb)**

欢迎！也许您是一位对Chisel感兴趣的学生，或者您是一位经验丰富的硬件设计专家，被经理要求探索Chisel作为新的HDL替代方案。无论哪种情况，如果您是Chisel的新手，您都希望尽快了解它为何如此受欢迎。不用再找了 - 让我们看看Chisel能提供什么！

## 设置
在开始之前，我们需要下载并导入演示所需的依赖项。

**请通过按键盘上的SHIFT+ENTER或菜单中的运行按钮来运行以下两个单元格块**。

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}

## 硬件生成器：类型安全的RTL元编程

所有硬件描述语言都支持编写单个RTL设计实例 - Chisel也不例外。
事实上，大多数Verilog/VHDL数字逻辑设计都可以直接转录到Chisel中！
虽然Chisel提供了其他很棒的功能（我们稍后会介绍），但我们要强调的是，切换到Chisel的用户将保留与任何其他硬件语言完全相同程度的设计控制权。

以下是一个以FIR滤波器风格实现的3点移动平均示例。

<img src="images/demo_fir_filter.svg" width="512" />

Chisel提供与可综合Verilog类似的基础原语，并且*可以*这样使用！运行下一个单元格来声明我们的Chisel模块。

In [ ]:
// 3-point moving average implemented in the style of a FIR filter
class MovingAverage3(bitWidth: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt(bitWidth.W))
  })

  val z1 = RegNext(io.in) // Create a register whose input is connected to the argument io.in
  val z2 = RegNext(z1)    // Create a register whose input is connected to the argument z1

  io.out := (io.in * 1.U) + (z1 * 1.U) + (z2 * 1.U) // `1.U` is an unsigned literal with value 1
}

定义 `class MovingAverage3` 后，让我们实例化它并查看其结构：

In [ ]:
// same 3-point moving average filter as before
visualize(() => new MovingAverage3(8))

在这个Chisel实例的可视化中，左侧是输入，金色的是z1和z2寄存器。寄存器和io_in都乘以它们的系数，然后依次相加。`tail` 和 `bits` 元素用于防止加法结果增长。

您现在可能会问："哦，很好 - 您可以在Chisel中做Verilog能做的事情，但为什么我要使用Chisel呢？"

我们很高兴您这么问！Chisel的真正力量来自于创建**生成器，而不是实例**的能力。假设我们不仅想要一个 `MovingAverage3` 模块，还想要创建一个通用的 `FIRFilter` 模块，该模块由系数列表参数化。

下面我们将 `MovingAverage3` 重写为接受系数序列。系数的数量将决定滤波器的大小。

In [ ]:
// Generalized FIR filter parameterized by the convolution coefficients
class FirFilter(bitWidth: Int, coeffs: Seq[UInt]) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt())
  })
  // Create the serial-in, parallel-out shift register
  val zs = Reg(Vec(coeffs.length, UInt(bitWidth.W)))
  zs(0) := io.in
  for (i <- 1 until coeffs.length) {
    zs(i) := zs(i-1)
  }

  // Do the multiplies
  val products = VecInit.tabulate(coeffs.length)(i => zs(i) * coeffs(i))

  // Sum up the products
  io.out := products.reduce(_ +& _)
}

现在通过在实例化过程中更改 `coeffs` 参数，我们的 `FIRFilter` 模块可用于实例化无限多个不同的硬件模块！下面我们创建三个不同的 `FIRFilter` 实例

In [ ]:
// same 3-point moving average filter as before
visualize(() => new FirFilter(8, Seq(1.U, 1.U, 1.U)))

In [ ]:
// 1-cycle delay as a FIR filter
visualize(() => new FirFilter(8, Seq(0.U, 1.U)))

In [ ]:
// 5-point FIR filter with a triangle impulse response
visualize(() => new FirFilter(8, Seq(1.U, 2.U, 3.U, 2.U, 1.U)))

没有这种强大的参数化功能，我们将需要更多的模块定义，可能每个FIR滤波器都需要一个。理想情况下，我们希望我们的生成器能够：(1) 可组合，(2) 功能强大，(3) 对生成的设计提供细粒度控制。

Chisel的优势在于您如何使用它，而不是语言本身。
如果您决定编写实例而不是生成器，您将看到Chisel相对于Verilog的优势较少。
但如果您花时间学习如何编写生成器，那么Chisel的强大功能将变得明显，您会意识到您再也无法回到编写Verilog的时代。
学习编写生成器很困难，但我们希望本教程能为您成为更好的硬件设计师、程序员和思考者铺平道路！

---
# 全部完成！

[返回顶部。](#top)